# Construction Site Safety Monitoring with YOLO11 (25 Classes)

This notebook reproduces the full multi-class object detection experimentation workflow on the public Roboflow **Construction Site Safety v30** dataset (**CC BY 4.0**, 717 images, 25 PPE / site / machinery classes):

1. **Dataset Download**: Fetch v30 in `yolov11` format via `ROBOFLOW_API_KEY` (loaded from `.env` or environment).
2. **Baseline Experiments**: Train and compare **YOLO11s** vs **YOLO11m** (50 epochs, `imgsz=640`).
3. **Custom Data Augmentation**: Train **YOLO11s** with domain-aware augmentation (`configs/train_augment.yaml`).
4. **Hyperparameter Tuning**: Optional evolutionary search via `model.tune(...)`.
5. **Comparative Evaluation & Sample Visualization**: Evaluate all models on the held-out test split, plot comparison metrics and normalized confusion matrices, and inspect Ground Truth vs Predicted boxes.

> **Reproducibility Note vs. Historical `v1.0.0` Checkpoint**: During repository cleanup for standalone local execution, two issues from the original cloud-assisted experiments were fixed: (1) `model.tune(...)` now explicitly sets `val=True` and passes `data=str(data_yaml)` so Ultralytics computes validation fitness at every iteration, and (2) `configs/train_augment.yaml` removes local paths and sets `pretrained: true` for COCO transfer learning. Re-training locally with these corrected settings may therefore produce slightly different metrics than the historical `11s_aug` checkpoint distributed in Release `v1.0.0`.

In [ ]:
import os
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as IPyImage, display
from roboflow import Roboflow
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"
AUGMENT_CFG = PROJECT_ROOT / "configs" / "train_augment.yaml"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

# Load optional .env file from project root without overriding existing shell env vars.
env_file = PROJECT_ROOT / ".env"
if env_file.is_file():
    for raw_line in env_file.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            k = k.strip()
            if not k:
                continue
            v = v.strip()
            if len(v) >= 2 and v[0] == v[-1] and v[0] in {"'", '"'}:
                v = v[1:-1]
            os.environ.setdefault(k, v)

print("Project root:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Download Dataset (Roboflow Universe — Construction Site Safety v30, CC BY 4.0)

Copy `.env.example` to `.env` and set `ROBOFLOW_API_KEY`, or export it in your shell before running this cell.

In [ ]:
api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key or api_key == "your_roboflow_api_key_here":
    raise RuntimeError("Set a valid ROBOFLOW_API_KEY in .env or your environment before downloading.")

dataset_dir = DATASETS_DIR / "construction-site-safety-v30"
rf = Roboflow(api_key=api_key)
dataset = (
    rf.workspace("roboflow-universe-projects")
    .project("construction-site-safety")
    .version(30)
    .download("yolov11", location=str(dataset_dir))
)

candidates = list(Path(dataset.location).rglob("data.yaml"))
if not candidates:
    raise FileNotFoundError(f"Could not locate data.yaml under {dataset.location}")
data_yaml = candidates[0]
print("Using data.yaml:", data_yaml)

## 2. Experiment 1 — YOLO11s Baseline (50 Epochs)

In [ ]:
model_11s = YOLO("yolo11s.pt")
res_11s = model_11s.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name="yolo11s_baseline",
)
weights_11s = Path(res_11s.save_dir) / "weights" / "best.pt"
print("YOLO11s best weights:", weights_11s)

## 3. Experiment 2 — YOLO11m Baseline (50 Epochs)

In [ ]:
model_11m = YOLO("yolo11m.pt")
res_11m = model_11m.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name="yolo11m_baseline",
)
weights_11m = Path(res_11m.save_dir) / "weights" / "best.pt"
print("YOLO11m best weights:", weights_11m)

## 4. Experiment 3 — YOLO11s with Custom Construction-Site Augmentation

Uses `configs/train_augment.yaml` (outdoor HSV lighting jitter, scale/translate/shear/perspective, horizontal flip, `flipud=0.0`, mosaic + mixup, COCO transfer learning enabled via `pretrained: true`).

In [ ]:
model_11s_aug = YOLO("yolo11s.pt")
res_11s_aug = model_11s_aug.train(
    data=str(data_yaml),
    cfg=str(AUGMENT_CFG),
    project=str(RUNS_DIR),
    name="yolo11s_augmented",
)
weights_11s_aug = Path(res_11s_aug.save_dir) / "weights" / "best.pt"
print("YOLO11s + Aug best weights:", weights_11s_aug)

## 5. Optional — Hyperparameter Tuning (`model.tune`)

Kept behind a guard (`RUN_TUNING = False`) because multi-iteration evolutionary search is compute-intensive. Unlike the earlier exploratory run (where `val=False` prevented fitness scoring), this cell explicitly sets `val=True` and `data=str(data_yaml)`.

In [ ]:
RUN_TUNING = False

if RUN_TUNING:
    search_space = {
        "lr0": (1e-5, 1e-1),
        "degrees": (0.0, 20.0),
        "momentum": (0.6, 0.98),
        "weight_decay": (0.0, 0.001),
        "warmup_epochs": (0.0, 5.0),
        "warmup_momentum": (0.0, 0.95),
        "box": (1.0, 20.0),
        "cls": (0.2, 4.0),
        "hsv_h": (0.0, 0.05),
        "hsv_s": (0.0, 0.9),
        "hsv_v": (0.0, 0.9),
        "translate": (0.0, 0.3),
        "scale": (0.0, 0.8),
        "shear": (0.0, 5.0),
        "perspective": (0.0, 0.001),
        "flipud": (0.0, 0.0),
        "fliplr": (0.0, 1.0),
        "mosaic": (0.0, 1.0),
        "mixup": (0.0, 0.4),
    }
    tuner_model = YOLO("yolo11s.pt")
    tuner_model.tune(
        data=str(data_yaml),
        epochs=30,
        iterations=50,
        optimizer="AdamW",
        space=search_space,
        plots=True,
        save=False,
        val=True,
        project=str(RUNS_DIR),
        name="tune_yolo11s",
    )

## 6. Held-Out Test Split Evaluation & Model Comparison

In [ ]:
def evaluate_checkpoint(weights_path: Path, data_path: Path, split: str = "test") -> dict:
    model = YOLO(str(weights_path))
    res = model.val(data=str(data_path), split=split, project=str(RUNS_DIR), name=f"eval_{weights_path.parent.parent.name}_{split}")
    d = res.results_dict
    return {
        "precision": float(d["metrics/precision(B)"]),
        "recall": float(d["metrics/recall(B)"]),
        "mAP50": float(d["metrics/mAP50(B)"]),
        "mAP50-95": float(d["metrics/mAP50-95(B)"]),
        "save_dir": Path(res.save_dir),
    }

checkpoints = {
    "YOLO11s (Baseline)": weights_11s,
    "YOLO11m (Baseline)": weights_11m,
    "YOLO11s + Custom Aug": weights_11s_aug,
}

test_results = {name: evaluate_checkpoint(path, data_yaml, split="test") for name, path in checkpoints.items()}
for name, m in test_results.items():
    print(f"{name:24} | P={m['precision']:.3f} R={m['recall']:.3f} mAP@50={m['mAP50']:.3f} mAP@50-95={m['mAP50-95']:.3f}")

In [ ]:
metric_keys = ["precision", "recall", "mAP50", "mAP50-95"]
metric_labels = ["Precision", "Recall", "mAP@50", "mAP@50-95"]
x = np.arange(len(metric_keys))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
for idx, (model_name, metrics) in enumerate(test_results.items()):
    vals = [metrics[k] for k in metric_keys]
    bars = ax.bar(x + (idx - 1) * width, vals, width, label=model_name)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom", fontsize=8)

ax.set_ylabel("Score")
ax.set_ylim(0, 0.9)
ax.set_title("Construction Site Safety — Held-Out Test Split Comparison (25 Classes)")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

# Display normalized confusion matrix for the top-performing test checkpoint
best_name = max(test_results, key=lambda n: test_results[n]["mAP50-95"])
cm_path = test_results[best_name]["save_dir"] / "confusion_matrix_normalized.png"
if cm_path.is_file():
    print(f"Normalized confusion matrix ({best_name}):")
    display(IPyImage(filename=str(cm_path), width=900))

## 7. Ground Truth vs. Predicted Bounding Boxes (Sample of Test Images)

In [ ]:
def plot_sample_predictions(weights_path: Path, data_root: Path, num_samples: int = 4, conf: float = 0.35) -> None:
    model = YOLO(str(weights_path))
    names = model.names

    test_img_dirs = [data_root / "test" / "images", data_root / "images" / "test"]
    img_dir = next((d for d in test_img_dirs if d.is_dir()), None)
    if img_dir is None:
        raise FileNotFoundError(f"Test image directory not found under {data_root}")
    lbl_dir = img_dir.parent / "labels" if img_dir.name == "images" else img_dir.parent.parent / "labels" / "test"

    image_files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})[:num_samples]
    if not image_files:
        raise FileNotFoundError(f"No supported test images found in {img_dir}")
    fig, axes = plt.subplots(len(image_files), 3, figsize=(16, 4.8 * len(image_files)))
    if len(image_files) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, img_path in enumerate(image_files):
        orig_bgr = cv2.imread(str(img_path))
        if orig_bgr is None:
            continue
        h, w = orig_bgr.shape[:2]

        gt_bgr = orig_bgr.copy()
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.is_file():
            for line in lbl_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
                    x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
                    cv2.rectangle(gt_bgr, (x1, y1), (x2, y2), (255, 110, 30), 2)
                    cv2.putText(gt_bgr, names.get(cls_id, str(cls_id)), (x1, max(16, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 110, 30), 2)

        pred_res = model.predict(str(img_path), conf=conf, verbose=False)[0]
        pred_bgr = pred_res.plot()

        for col, (img, title) in enumerate([
            (orig_bgr, f"Original ({img_path.name})"),
            (gt_bgr, "Ground Truth"),
            (pred_bgr, f"Predicted (conf>={conf:.2f})"),
        ]):
            axes[row, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[row, col].set_title(title, fontsize=10)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

plot_sample_predictions(checkpoints[best_name], data_yaml.parent, num_samples=4)

## 8. Export Best Checkpoint for CLI Inference

In [ ]:
target_weights = PROJECT_ROOT / "weights" / "best.pt"
target_weights.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(checkpoints[best_name], target_weights)
print(f"Exported '{best_name}' checkpoint to {target_weights}")